# Anexo técnico: leer y reproducir el benchmark parcial

Este cuaderno verifica el inventario y presenta las tablas completas ya medidas. No vuelve a ajustar modelos al abrirlo. La reproducción desde cero está en `reproduce.py`, incluido en esta descarga: ejecuta calentamiento y tres procesos independientes secuenciales, compara sus resultados y calcula los intervalos predeclarados.

El mínimo está en `minimo.ipynb`. Este anexo añade conceptos de inventario, cobertura, métricas, incertidumbre y procedencia; no representa el flujo mínimo del modelador.


In [1]:
import json
from pathlib import Path

import pandas as pd
from h10_benchmark.contract import verify_coverage, verify_sources

verify_sources(Path("."))
rows = json.loads(Path("results/results.json").read_text(encoding="utf-8"))
verify_coverage(rows)

{'total': 28, 'executed': 28, 'failed': 0, 'pending': 0}

## Todas las combinaciones admitidas

Cuatro carteras por siete recetas: 28. German y Taiwan suplementario usan una evaluación exterior de grupos X exactos, 25% y semilla 20261016; el target y el ID se excluyen de X. Los sintéticos conservan OOT 2024Q2. Sólo Desarrollo ajusta; Holdout es diagnóstico. No se acredita independencia entre clientes. No se seleccionó ninguna receta por estos resultados.

Las tablas mantienen fallos y pendientes en el denominador. LC, HC, PAKDD y R siguen pendientes de la capa C y no forman parte de las 28 combinaciones admitidas.


In [2]:
coverage = pd.DataFrame(rows)[["case", "recipe", "status", "error"]]
coverage

,case,recipe,status,error
0,german,fabrica,ejecutado,None
1,german,sin_stepwise,ejecutado,None
2,german,cinco_bins,ejecutado,None
3,german,monotonia_auto,ejecutado,None
4,german,optbinning_mip,ejecutado,None
5,german,scorecardpy,ejecutado,None
6,german,skorecard_mip,ejecutado,None
7,taiwan,fabrica,ejecutado,None
8,taiwan,sin_stepwise,ejecutado,None
9,taiwan,cinco_bins,ejecutado,None


## Métricas y tiempos completos

PD creciente de riesgo. AUC, Gini=2AUC−1, KS, Brier, log-loss, PD media, tasa observada y PSI con cortes de Desarrollo. Tiempos en segundos: mediana y rango de tres procesos secuenciales; calentamiento excluido. `fit_seconds` es el pipeline bayesrisk sin el paso de informe; en referentes es su fit. `report_seconds` mide HTML/Word, no Excel ni el verificador del arnés. No son costes funcionalmente equivalentes entre herramientas.


In [3]:
metrics = pd.read_csv("aggregate/metrics.csv", float_precision="round_trip")
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(metrics)

,case,recipe,status,error,n,events,auc,gini,ks,brier,logloss,observed,mean_pd,psi,psi_bins,variables_finales,variables_entrada,fit_seconds_median,fit_seconds_min,fit_seconds_max,apply_seconds_median,apply_seconds_min,apply_seconds_max,report_seconds_median,report_seconds_min,report_seconds_max
0,german,fabrica,ejecutado,NaN,250,79,0.783256,0.566511,0.528463,0.176774,0.518292,0.316000,0.294721,0.030979,10,11,20,2.731313,2.682875,2.902826,0.379110,0.375258,0.399686,5.107536,5.103683,5.491307
1,german,sin_stepwise,ejecutado,NaN,250,79,0.787549,0.575098,0.530387,0.174993,0.514827,0.316000,0.293423,0.035762,10,14,20,1.288412,1.261179,1.359260,0.416152,0.409121,0.432099,5.043446,5.004487,5.368301
2,german,cinco_bins,ejecutado,NaN,250,79,0.784625,0.569250,0.534310,0.175958,0.515932,0.316000,0.295019,0.027793,10,11,20,2.408341,2.399207,2.460493,0.463931,0.372269,0.487954,5.028116,4.973252,5.039769
3,german,monotonia_auto,ejecutado,NaN,250,79,0.789437,0.578873,0.485528,0.174233,0.513096,0.316000,0.298065,0.018642,10,10,20,2.471153,2.408939,2.557601,0.381538,0.378699,0.396958,4.968439,4.797223,5.101882
4,german,optbinning_mip,ejecutado,NaN,250,79,0.802280,0.604560,0.548893,0.168651,0.496691,0.316000,0.296749,0.057407,10,20,20,0.299909,0.286390,0.301862,0.017145,0.016664,0.017637,0.000000,0.000000,0.000000
5,german,scorecardpy,ejecutado,NaN,250,79,0.772892,0.545784,0.461248,0.182249,0.534624,0.316000,0.297016,0.059110,10,20,20,2.948284,2.754862,3.069620,0.138802,0.135765,0.149215,0.000000,0.000000,0.000000
6,german,skorecard_mip,ejecutado,NaN,250,79,0.803020,0.606040,0.543045,0.168253,0.495607,0.316000,0.297276,0.035499,10,20,20,1.552372,1.533368,1.636152,0.101145,0.096805,0.104677,0.000000,0.000000,0.000000
7,taiwan,fabrica,ejecutado,NaN,7500,1675,0.766529,0.533058,0.411228,0.139062,0.441906,0.223333,0.218254,0.001015,10,10,23,17.770214,17.759762,18.457734,2.388614,2.287723,2.512894,6.621247,6.374760,6.700636
8,taiwan,sin_stepwise,ejecutado,NaN,7500,1675,0.767566,0.535132,0.408668,0.138936,0.441477,0.223333,0.218215,0.000343,10,14,23,14.028611,13.715342,16.002991,2.990588,2.987102,3.151644,6.918181,6.802761,7.206478
9,taiwan,cinco_bins,ejecutado,NaN,7500,1675,0.766842,0.533684,0.412722,0.139041,0.441782,0.223333,0.218246,0.000883,10,10,23,17.790329,17.328737,18.009589,2.299375,2.191242,2.350741,6.621798,6.355419,6.626392


## Incertidumbre pareada

1.000 remuestreos de grupos completos del test, con reemplazo y semilla 20261016. Diferencia de AUC respecto de fábrica. Intervalo exploratorio condicionado a modelos y partición: no estima variación de entrenamiento ni corrige comparaciones múltiples. Una réplica de una sola clase impide publicar el intervalo; el contador de inválidas se conserva.


In [4]:
intervals = pd.read_json("aggregate/bootstrap.json")
intervals

,case,recipe,replicates,invalid,groups,seed,delta_auc,interval_95,limitation
0,german,sin_stepwise,1000,0,250,20261016,0.004293,"[-0.0040868826912320005, 0.013577331892894002]",Condicionado a modelos y partición; sin varian...
1,german,cinco_bins,1000,0,250,20261016,0.001369,"[-0.0029641620015810003, 0.0055617679961360005]",Condicionado a modelos y partición; sin varian...
2,german,monotonia_auto,1000,0,250,20261016,0.006181,"[-0.012938887548262001, 0.027026433949529003]",Condicionado a modelos y partición; sin varian...
3,german,optbinning_mip,1000,0,250,20261016,0.019024,"[0.0018707663264580002, 0.036382840173672]",Condicionado a modelos y partición; sin varian...
4,german,scorecardpy,1000,0,250,20261016,-0.010363,"[-0.034800563922153, 0.015076086241399]",Condicionado a modelos y partición; sin varian...
5,german,skorecard_mip,1000,0,250,20261016,0.019765,"[0.0010018690543070002, 0.037743267588342004]",Condicionado a modelos y partición; sin varian...
6,taiwan,sin_stepwise,1000,0,7486,20261016,0.001037,"[-1.9509952827898022e-05, 0.001966785230722]",Condicionado a modelos y partición; sin varian...
7,taiwan,cinco_bins,1000,0,7486,20261016,0.000313,"[-0.000200181714715, 0.000824817829918]",Condicionado a modelos y partición; sin varian...
8,taiwan,monotonia_auto,1000,0,7486,20261016,0.000091,"[-0.001977883129725, 0.002387708450899]",Condicionado a modelos y partición; sin varian...
9,taiwan,optbinning_mip,1000,0,7486,20261016,0.005074,"[0.001357412372063, 0.008847490093935]",Condicionado a modelos y partición; sin varian...


## Artefactos, configuración y reproducción

`artifacts/` contiene HTML, Word y Excel de las 16 corridas bayesrisk. `results/tables/` conserva las tablas de los tres referentes y los resúmenes. `expected/` contiene el contenido esperado capturado de los Studies propios, sin distribuir pickles. `prepared/` conserva fuentes transformadas, índices, grupos, muestras y manifiestos; `cache/` contiene sólo las dos fuentes UCI autorizadas. El mapa `provenance.json` distingue bytes originales y derivados con rutas portables.

Las cifras de este cuaderno pertenecen a la evaluación exterior. No se presentan como métricas del Holdout interior ni se construyen Studies de terceros para Comparison.

Para reproducir después de instalar el entorno y el wheel candidato exacto descritos en la guía, inicia un proceso con `PYTHONHASHSEED=0` y ejecuta:

```text
python reproduce.py
```

El script crea `reproduction/`, rechaza sobrescribir una reproducción previa, comprueba fuentes y conserva las 28 filas incluso ante fallos. La falta de red con cache ausente o una fuente alterada bloquea la ejecución; no se interpreta como éxito. No exige credenciales.
